# Analiza dialogowa scenariusza: synteza i prezentacja

## Wersja studencka — moduł 4: SYNTEZA

W poprzednich modułach zebrałeś trzy warstwy analizy:
- **Sieć współwystąpień** — kto z kim jest w scenach (`graf_postaci.graphml`)
- **Głos i płeć** — kto mówi i ile (`dialogi.csv`, `postacie.csv`)
- **Emocje** — z jakim ładunkiem emocjonalnym (`emocje.csv`, `emocje_postaci.csv`)

Ten notatnik łączy wszystkie trzy warstwy w jedno, gotowe do prezentacji wyjście. Efektem końcowym będzie plik `bundle.json` zawierający pełne dane o filmie, wzbogacony plik sieci (`graf_postaci_wzbogacony.graphml`) i archiwum ZIP do pobrania. Na tej podstawie wygenerujesz — za pomocą gotowego promptu — **pełną prezentację HTML** w estetyce dopasowanej do Twojego filmu.

## Jak pracować z tym notatnikiem

1. Upewnij się, że masz w tym samym środowisku wszystkie pliki z poprzednich modułów: `graf_postaci.graphml`, `dialogi.csv`, `postacie.csv`, `emocje.csv`, `emocje_postaci.csv`.
2. Skopiuj prompt z bieżącego kroku do modelu AI, wklej kod, uruchom i porównaj z sekcją **Po uruchomieniu powinieneś zobaczyć**.
3. Etap 5 nie ma komórki kodu — to krok wykonywany poza notatnikiem, w oknie czatu z modelem AI.

---
## Etap 1: Wczytanie i inspekcja danych

Zaczynamy od załadowania wszystkich plików i weryfikacji, że wszystko jest na miejscu. Ponieważ każdy student generował kod trochę inaczej, nazwy kolumn mogą się różnić — dlatego najpierw sprawdzamy, co dokładnie mamy.

### Krok 1A. Wczytanie plików i raport o kolumnach

#### Cel i sens analityczny

Zanim scalimy dane z trzech modułów, musimy wiedzieć, jakie kolumny mamy w każdym pliku. To typowy krok integracyjny: dane z różnych źródeł rzadko pasują do siebie „od razu".

#### Prompt dla modelu

```text
Kontekst:
Masz pięć plików z poprzednich modułów analizy scenariusza filmowego.

Wejście:
Pliki: `graf_postaci.graphml`, `dialogi.csv`, `postacie.csv`, `emocje.csv`, `emocje_postaci.csv`.

Zadanie:
Wczytaj wszystkie pięć plików. Dla każdego pliku CSV wypisz: liczbę wierszy i dokładne nazwy kolumn. Dla pliku GraphML wypisz: liczbę węzłów i krawędzi. Na końcu sprawdź, czy kolumna z nazwą postaci istnieje we wszystkich czterech plikach CSV i czy nazwy postaci są spójne między plikami (pokaz kilka przykładów z każdego pliku).

Pokaż wynik:
- tabelkę: plik | liczba wierszy/węzłów | kolumny lub wymiary,
- porównanie 5 przykładowych nazw postaci z postacie.csv vs emocje_postaci.csv.

Warunek poprawności:
Nazwy postaci powinny wyglądać tak samo (wielkość liter, spacje) w obu plikach. Jeśli się różnią, zaznacz to — będzie trzeba ujednolicić je w kolejnym kroku.

Jeśli wystąpi błąd:
Wyświetl, którego pliku nie udało się wczytać i podaj możliwą przyczynę (zła ścieżka, kodowanie, zły format).

Nie rób jeszcze:
Nie scalaj danych ani nie modyfikuj grafu.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Listę plików z liczbą wierszy i nazwami kolumn.
- Potwierdzenie, że plik GraphML ma węzły i krawędzie.
- Porównanie nazw postaci między plikami CSV — ideally identyczne.

> **Jeśli nazwy postaci się różnią** (np. spacje, wielkość liter), poproś model o krok normalizujący przed przejściem dalej: *„Dopasuj nazwy postaci między plikami: usuń zbędne spacje i ujednolicaj wielkość liter"*.

---
## Etap 2: Wzbogacenie sieci i przygotowanie pakietu danych

To główny krok tego modułu. Dołączamy do każdego węzła sieci dane z poprzednich modułów (płeć, liczba słów, dominująca emocja, średnia walencja) i obliczamy agregaty potrzebne do wykresów w prezentacji.

### Krok 2A. Wzbogacenie grafu i zapis bundle.json

#### Cel i sens analityczny

Wzbogacony graf łączy wszystko: wiesz już nie tylko, kto z kim rozmawia, ale czyim głosem dominuje scena, jaką emocją jest nacechowany i jakiej jest płci. Plik `bundle.json` zbiera te dane w formacie gotowym do wklejenia do modelu generującego prezentację.

#### Prompt dla modelu

```text
Kontekst:
Masz wczytane: graf sieci (networkx), tabelę postacie.csv (postać, płeć, liczba_słów lub podobna kolumna), tabelę emocje_postaci.csv (postać, dominująca emocja, średnia walencja) oraz tabelę emocje.csv (kwestie z numerem sceny, walencją i emocją). Nazwy kolumn mogą się różnić od podanych — użyj tych, które faktycznie są w plikach.

Wejście:
Wszystkie wczytane dane z Kroku 1A.

Zadanie:
Wykonaj trzy rzeczy:

1. WZBOGAĆ GRAF: dla każdego węzła grafu dołącz atrybuty z postacie.csv i emocje_postaci.csv (płeć, liczba słów, dominująca emocja, średnia walencja). Węzły, dla których nie ma danych, oznacz jako "?" / 0 / "neutralna" / 0.0. Zapisz wzbogacony graf do pliku `graf_postaci_wzbogacony.graphml`.

2. OBLICZ AGREGATY (potrzebne do wykresów):
   a. Ranking mowy: lista top 15 postaci wg liczby słów z płcią i procentem łącznej mowy.
   b. Mowa wg płci: łączna liczba słów dla K, M i ? oraz ich procent.
   c. Łuk emocjonalny: dla każdej sceny oblicz średnią walencję wszystkich kwestii, następnie wygładź ruchomym oknem o szerokości 5. Wynik to lista par (numer_sceny, wygładzona_walencja).
   d. Emocja × płeć: dla K i M osobno policz, ile kwestii przypada na każdą z 6 emocji, i przelicz na procenty (żeby K i M były porównywalne mimo różnej liczby kwestii).

3. ZAPISZ BUNDLE: zapisz słownik z poniższą strukturą do pliku `bundle.json` (UTF-8, wcięcia=2):
{
  "top_speakers": [{"name": ..., "gender": ..., "word_count": ..., "pct_of_total": ...}],
  "speech_by_gender": {"K": {"total_words": ..., "pct": ...}, "M": {...}, "?": {...}},
  "emotion_arc": [{"scene": ..., "avg_valence": ..., "rolling_valence": ...}],
  "emotion_by_gender": {
    "K": {"radość": ..., "smutek": ..., "złość": ..., "strach": ..., "zaskoczenie": ..., "neutralna": ...},
    "M": {tak samo}
  },
  "network": {
    "nodes": [{"id": ..., "gender": ..., "word_count": ..., "scene_count": ..., "dominant_emotion": ..., "avg_valence": ...}],
    "edges": [{"source": ..., "target": ..., "weight": ...}]
  }
}

Pokaż wynik:
- potwierdzenie zapisu obu plików (graf + bundle.json) z ich rozmiarami,
- 5 przykładowych węzłów z wzbogaconego grafu (postać + nowe atrybuty),
- pierwsze i ostatnie 3 wiersze łuku emocjonalnego.

Warunek poprawności:
Suma procentów speech_by_gender powinna wynosić 100%. Procenty emocji dla K powinny sumować się do 100%, tak samo dla M. Łuk emocjonalny powinien zawierać tyle wierszy, ile jest scen (lub nieznacznie mniej, jeśli część scen nie ma kwestii).

Jeśli wystąpi błąd:
Wypisz, na którym kroku (wzbogacanie grafu / agregaty / bundle) wystąpił problem i pokaż treść błędu.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Potwierdzenie zapisu `graf_postaci_wzbogacony.graphml` i `bundle.json`.
- Kilka węzłów z atrybutami: płeć, liczba słów, emocja, walencja.
- Pierwsze i ostatnie wpisy łuku emocjonalnego z wartościami walencji.

---
## Etap 3: Podgląd sieci wzbogaconej

To czwarty wykres-bohater całego modułu: ta sama sieć co w pierwszym module, ale teraz każdy węzeł ma kolor płci i rozmiar proporcjonalny do ilości mowy. Pokazuje jednym obrazem, kto ma głos w tej sieci — i czyją sieć to jest.

### Krok 3A. Wykres sieci z atrybutami głosu i płci

#### Cel i sens analityczny

Wizualizacja wzbogaconej sieci to pomost między analizą sieciową a dialogową. Wcześniej widziałeś, kto z kim *bywa*. Teraz widzisz, kto z kim *bywa i ile mówi*. Postać centralnie połączona, ale małych rozmiarów — to ktoś, kto dużo *obserwuje*, ale rzadko *mówi*.

#### Prompt dla modelu

```text
Kontekst:
Masz wzbogacony graf wczytany z `graf_postaci_wzbogacony.graphml` (lub z danych obliczonych w poprzednim kroku). Węzły mają atrybuty: gender (K/M/?), word_count, dominant_emotion, avg_valence. Krawędzie mają atrybut weight.

Wejście:
Wzbogacony graf z poprzedniego kroku.

Zadanie:
Narysuj graf sieci z następującymi ustawieniami wizualnymi:
- Rozmiar węzła proporcjonalny do word_count (przeskaluj tak, żeby najmniejszy był widoczny, a największy nie dominował nad resztą).
- Kolor węzła według płci: K = jeden wyraźny kolor, M = drugi, ? = szary.
- Grubość krawędzi proporcjonalna do wagi (min 0.3, max 3 px).
- Etykiety nazw postaci tylko dla 12 węzłów o największym word_count (żeby nie zaśmiecać obrazu).
- Układ Fruchterman-Reingold (spring_layout) z seed=42 dla powtarzalności.
- Dodaj tytuł i legendę kolorów płci.
Zapisz wykres jako PNG o rozdzielczości 150 dpi do pliku `siec_wzbogacona.png`.

Pokaż wynik:
- wykres bezpośrednio w notatniku,
- potwierdzenie zapisu pliku PNG.

Warunek poprawności:
Węzły o dużym word_count powinny być wyraźnie większe od drobnych postaci. Jeśli sieć jest bardzo gęsta i nieczytelna, ogranicz wyświetlanie do węzłów z co najmniej 1 kwestią dialogową i zaznacz to w tytule.

Jeśli wystąpi błąd:
Wyjaśnij, czy problem dotyczy wczytywania grafu, brakujących atrybutów węzłów, czy rysowania.

Nie rób jeszcze:
Nie generuj prezentacji HTML.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Graf sieci z węzłami o zróżnicowanych rozmiarach i kolorach.
- Etykiety 12 najpopularniejszych mówców.
- Legendę kolorów płci i potwierdzenie zapisu PNG.

#### Pytanie interpretacyjne

Czy postacie z największą liczbą połączeń w sieci (te centralnie umieszczone) to też te, które najwięcej mówią (duże węzły)? Albo inaczej: czy są postacie *drobne* (mało mówią), a jednak *centralnie* osadzone w sieci? Co to może znaczyć narracyjnie?

---
## Etap 4: Archiwum do pobrania

Zbieramy wszystkie pliki wyjściowe w jedno miejsce i tworzymy archiwum ZIP gotowe do pobrania z Colab.

### Krok 4A. Zapis archiwum ZIP

#### Cel i sens analityczny

Prezentację HTML wygenerujesz w kolejnym kroku poza notatnikiem — potrzebujesz więc wszystkich plików lokalnie na swoim komputerze. ZIP jest wygodniejszy niż pobieranie plików po jednym.

#### Prompt dla modelu

```text
Kontekst:
Masz gotowe pliki wyjściowe z trzech modułów analizy.

Wejście:
Pliki do spakowania: `bundle.json`, `graf_postaci_wzbogacony.graphml`, `siec_wzbogacona.png`, `dialogi.csv`, `postacie.csv`, `emocje.csv`, `emocje_postaci.csv`. Jeśli któregoś pliku nie ma, pomiń go i wypisz ostrzeżenie.

Zadanie:
Utwórz archiwum ZIP o nazwie `analiza_[nazwa_pliku_graphml_bez_rozszerzenia].zip` (np. `analiza_graf_postaci_wzbogacony.zip`) zawierające wszystkie wymienione pliki. Następnie wyświetl link do pobrania z Colab oraz listę plików w archiwum z ich rozmiarami.

Pokaż wynik:
- link do pobrania archiwum ZIP,
- listę spakowanych plików z rozmiarami,
- ostrzeżenia o brakujących plikach (jeśli są).

Warunek poprawności:
Archiwum ZIP powinno zawierać przynajmniej: bundle.json, graf_postaci_wzbogacony.graphml. Pozostałe pliki są opcjonalne, ale ich brak warto odnotować.

Jeśli wystąpi błąd:
Wyjaśnij, którego pliku nie udało się spakować.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Link do pobrania archiwum ZIP.
- Listę plików w archiwum z rozmiarami.
- Ewentualne ostrzeżenia o brakujących plikach.

---
## Etap 5: Generowanie prezentacji HTML

Ostatni krok dzieje się **poza notatnikiem** — w oknie czatu z modelem AI. To ten sam wzorzec, który znasz z wcześniejszego modułu, gdy generowałeś interaktywny dashboard SNA.

### Jak to zrobić

1. **Pobierz** archiwum ZIP z poprzedniego kroku i rozpakuj je lokalnie.
2. **Otwórz** plik `prompts/prompt_dialogowy_deck.md` z repozytorium kursu — to gotowy prompt do generowania prezentacji.
3. **Skopiuj** cały tekst promptu (zaznacz od `Jesteś ekspertem...` do końca).
4. **Otwórz** Claude (claude.ai) lub Gemini i wklej prompt.
5. **Dołącz** do wiadomości jeden plik: `bundle.json`.
6. **Wyślij** i poczekaj na wygenerowanie pełnego pliku HTML.
7. **Zapisz** odpowiedź jako `[tytuł_filmu]_prezentacja.html` i otwórz w przeglądarce.

### Co zawiera wygenerowana prezentacja

Deck HTML to jeden plik z sześcioma slajdami nawigowanymi strzałkami klawiatury i przyciskami:

| Slajd | Zawartość |
|---|---|
| 1 | Tytuł — film, podtytuł „Głos, Płeć, Emocje" |
| 2 | Udział w dialogu × płeć (wykres słupkowy) |
| 3 | Łuk emocjonalny (wykres liniowy) |
| 4 | Emocje × płeć (wykres grupowany) |
| 5 | Sieć postaci — interaktywna (D3, węzły kolorowane płcią, rozmiar = mowa) |
| 6 | Interpretacja humanistyczna |

Estetyka i kolory są automatycznie dobierane do Twojego filmu na podstawie nazw postaci.

> **Jeśli plik HTML jest obcięty** (model nie dokończył generowania), poproś go: *„Kontynuuj od miejsca, gdzie się zatrzymałeś"* albo wygeneruj deck w kilku częściach, prosząc najpierw o slajdy 1–3, potem 4–6.

---
## Co dalej?

Masz teraz kompletny zestaw materiałów dla swojego filmu:

- `analiza_sieciowa_*.ipynb` + `graf_postaci.graphml` — sieć współwystąpień (moduł 1)
- `dialogi.csv` + `postacie.csv` — głos i płeć (moduł 2)
- `emocje.csv` + `emocje_postaci.csv` — emocje (moduł 3)
- `graf_postaci_wzbogacony.graphml` + `bundle.json` + `siec_wzbogacona.png` — synteza (moduł 4)
- `[film]_prezentacja.html` — gotowa prezentacja

**Możliwe dalsze kroki:**

- Porównaj swój film z wynikami innych studentów: czy filmy z tego samego gatunku mają podobny łuk emocjonalny? Podobny rozkład mowy według płci?
- Dodaj do sieci wymiar czasu — podziel krawędzie na dwie warstwy (pierwsza i druga połowa filmu) i sprawdź, jak struktura się zmienia.
- Sprawdź, czy postać o najwyższym betweenness centrality to też ta, która wypowiada najwięcej słów — albo zupełnie inna.